<a href="https://colab.research.google.com/github/Prezzo-K/Graph-Neural-Networks-for-Fraud-Detection/blob/classical-ml-experiments/notebooks/Traditional_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# import scikit learn and check version
import sklearn as sk

sk.__version__

'1.6.1'

In [3]:
# Workflow
# 1. Get the dataset, do preprocessing if any and split it into train and test set
# 2. Load various clasical ML models - Random Forest (RF), Extra Trees, Logistic Regression (LR), Support Vector Machine (SVM), XGBoost
# 3. Train and test them on the test set
# 3. Compare their Accuracy, Precision and Recall, F1-Score, Area Under the ROC Curve (AUC–ROC), Confusion Matrix


# Load Dataset

In [29]:
import pandas as pd

file_path = "/content/Fraud Detection Transactions Dataset.csv"

dataset  = pd.read_csv(filepath_or_buffer = file_path)
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 21 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Transaction_ID                50000 non-null  object 
 1   User_ID                       50000 non-null  object 
 2   Transaction_Amount            50000 non-null  float64
 3   Transaction_Type              50000 non-null  object 
 4   Timestamp                     50000 non-null  object 
 5   Account_Balance               50000 non-null  float64
 6   Device_Type                   50000 non-null  object 
 7   Location                      50000 non-null  object 
 8   Merchant_Category             50000 non-null  object 
 9   IP_Address_Flag               50000 non-null  int64  
 10  Previous_Fraudulent_Activity  50000 non-null  int64  
 11  Daily_Transaction_Count       50000 non-null  int64  
 12  Avg_Transaction_Amount_7d     50000 non-null  float64
 13  F

In [30]:
# Retrieve the feature and labels separate

X = dataset.loc[:,"Transaction_ID":"Is_Weekend"]
y = dataset.loc[:, "Fraud_Label"]

X.head(2)

,Transaction_ID,User_ID,Transaction_Amount,Transaction_Type,Timestamp,Account_Balance,Device_Type,Location,Merchant_Category,IP_Address_Flag,Previous_Fraudulent_Activity,Daily_Transaction_Count,Avg_Transaction_Amount_7d,Failed_Transaction_Count_7d,Card_Type,Card_Age,Transaction_Distance,Authentication_Method,Risk_Score,Is_Weekend
0,TXN_33553,USER_1834,39.79,POS,2023-08-14 19:30:00,93213.17,Laptop,Sydney,Travel,0,0,7,437.63,3,Amex,65,883.17,Biometric,0.8494,0
1,TXN_9427,USER_7875,1.19,Bank Transfer,2023-06-07 04:01:00,75725.25,Mobile,New York,Clothing,0,0,13,478.76,4,Mastercard,186,2203.36,Password,0.0959,0


In [6]:
y.head(2)

,Fraud_Label
0,0
1,1


In [7]:
y.value_counts()

,count
Fraud_Label,
0,33933
1,16067


In [31]:
# Standardize columns with continous values - Transaction_Amount, Account_Balance, Daily_Transaction_Count, Avg_Transaction_Amount_7d, Card_Age, Transaction_Distance, Risk_Score
# Encode these as one hot encoding - Transaction_Type, Device_Type, Location, Merchant_Category, Card_Type, Authentication_Method
# Leave Binary / Numeric columns as they are. Don't scale or encode - IP_Address_Flag, Previous_Fraudulent_Activity, - Is_Weekend

# Drop the Transaction_ID, User_ID as they don't add in classification
X.drop(columns = ["Transaction_ID", "User_ID"], inplace = True)

# Remove risk score feature to avoid data leakge
# this feature seems to do data leakage though it will be beneficial in real world scenarios
X.drop(columns = ["Risk_Score"], inplace = True) # experiment it on/off with different models

# Convert 'Timestamp' to datetime and extract various fields
X["Timestamp"] = pd.to_datetime(X["Timestamp"])
X["Hour"] = X["Timestamp"].dt.hour
X["Day"] = X["Timestamp"].dt.day
X["Month"] = X["Timestamp"].dt.month
X["DayOfWeek"] = X["Timestamp"].dt.dayofweek
# drop off timestamp col now
X.drop(columns = ["Timestamp"], inplace = True)

# Do one hot encoding
categorical_cols = [
    "Transaction_Type",
    "Device_Type",
    "Location",
    "Merchant_Category",
    "Card_Type",
    "Authentication_Method"
]
X = pd.get_dummies(X, columns = categorical_cols)
# Display the first few rows with the new 'Hour' column to confirm
X

# Scale the dataset # TODO

,Transaction_Amount,Account_Balance,IP_Address_Flag,Previous_Fraudulent_Activity,Daily_Transaction_Count,Avg_Transaction_Amount_7d,Failed_Transaction_Count_7d,Card_Age,Transaction_Distance,Is_Weekend,...,Merchant_Category_Restaurants,Merchant_Category_Travel,Card_Type_Amex,Card_Type_Discover,Card_Type_Mastercard,Card_Type_Visa,Authentication_Method_Biometric,Authentication_Method_OTP,Authentication_Method_PIN,Authentication_Method_Password
0,39.79,93213.17,0,0,7,437.63,3,65,883.17,0,...,False,True,True,False,False,False,True,False,False,False
1,1.19,75725.25,0,0,13,478.76,4,186,2203.36,0,...,False,False,False,False,True,False,False,False,False,True
2,28.96,1588.96,0,0,14,50.01,4,226,1909.29,0,...,True,False,False,False,False,True,True,False,False,False
3,254.32,76807.20,0,0,8,182.48,4,76,1311.86,0,...,False,False,False,False,False,True,False,True,False,False
4,31.28,92354.66,0,1,14,328.69,4,140,966.98,1,...,False,False,False,False,True,False,False,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,45.05,76960.11,0,0,2,389.00,3,98,1537.54,1,...,False,False,True,False,False,False,False,False,True,False
49996,126.15,28791.75,0,0,13,434.95,4,93,2555.72,0,...,False,False,False,False,False,True,True,False,False,False
49997,72.02,29916.41,0,1,1,369.15,2,114,4686.59,0,...,False,False,False,False,False,True,True,False,False,False
49998,64.89,67895.67,0,0,13,242.29,4,72,4886.92,0,...,False,False,False,True,False,False,True,False,False,False


In [24]:
# split the data into train and test set
from sklearn.model_selection import train_test_split

random_state = 42
test_size = 0.2

X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size = test_size,
                                                    shuffle = True,
                                                    stratify = y
                                                    )
len(X_train), len(y_train), len(X_test), len(y_test)

(40000, 40000, 10000, 10000)

In [10]:
# do some sanity check to ensure good splits of the classes in train and test sets
y_train.value_counts(), y_test.value_counts()

(Fraud_Label
 0    27146
 1    12854
 Name: count, dtype: int64,
 Fraud_Label
 0    6787
 1    3213
 Name: count, dtype: int64)

# Load Models And Train

## Random Forest

In [11]:
from sklearn.ensemble import RandomForestClassifier

# create the model
rf = RandomForestClassifier()
# fit the model on the data
rf.fit(X_train, y_train)

RandomForestClassifier()

In [12]:
preds_rf = rf.predict(X_test)

In [15]:
# metrics -  Accuracy, Precision and Recall, F1-Score, Area Under the ROC Curve (AUC–ROC), Confusion Matrix
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report

def evaluate_model(preds, y_true, model_name="Model"):
    accuracy = accuracy_score(preds, y_true)
    f1 = f1_score(preds, y_true)
    precision = precision_score(preds, y_true)
    recall = recall_score(preds, y_true)
    report = classification_report(preds, y_true)

    print(f"\n{model_name} Metrics:")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"F1-Score: {f1:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print("\nClassification Report:\n", report)
    # return accuracy, f1, precision, recall, report

evaluate_model(preds_rf, y_test, model_name="Extra Trees Classifier")



Extra Trees Classifier Metrics:
Accuracy: 0.8772
F1-Score: 0.7638
Precision: 0.6178
Recall: 1.0000

Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.85      0.92      8015
           1       0.62      1.00      0.76      1985

    accuracy                           0.88     10000
   macro avg       0.81      0.92      0.84     10000
weighted avg       0.92      0.88      0.89     10000



## Extra Trees

In [18]:
from sklearn.ensemble import ExtraTreesClassifier

# load extra tree classifier
ext_clf = ExtraTreesClassifier()
# fit it in the data
ext_clf.fit(X_train, y_train)

# test on the test set
preds_ext_clf = ext_clf.predict(X_test)

# Evaluate the Extra Trees Classifier
evaluate_model(preds_ext_clf, y_test, model_name="Extra Trees Classifier")


Extra Trees Classifier Metrics:
Accuracy: 0.8771
F1-Score: 0.7636
Precision: 0.6178
Recall: 0.9995

Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.85      0.92      8014
           1       0.62      1.00      0.76      1986

    accuracy                           0.88     10000
   macro avg       0.81      0.92      0.84     10000
weighted avg       0.92      0.88      0.89     10000



## XGBoost

In [19]:
!pip install -q xgboost

In [20]:
from xgboost import XGBClassifier

# load extra tree classifier
xg_bst = XGBClassifier()
# fit it in the data
xg_bst.fit(X_train, y_train)

# test on the test set
xg_bst_preds = xg_bst.predict(X_test)

# Evaluate the Extra Trees Classifier
evaluate_model(xg_bst_preds, y_test, model_name="XGBoost Classifier")


XGBoost Classifier Metrics:
Accuracy: 0.8730
F1-Score: 0.7580
Precision: 0.6190
Recall: 0.9774

Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.85      0.91      7965
           1       0.62      0.98      0.76      2035

    accuracy                           0.87     10000
   macro avg       0.81      0.91      0.84     10000
weighted avg       0.92      0.87      0.88     10000



## Logistic Regression

In [21]:
from sklearn.linear_model import LogisticRegression

# Load Logistic Regression
lr = LogisticRegression()
# Fit it to the data
lr.fit(X_train, y_train)

# Test on the test set
preds_lr = lr.predict(X_test)

# Evaluate Logistic Regression
evaluate_model(preds_lr, y_test, model_name="Logistic Regression")

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



Logistic Regression Metrics:
Accuracy: 0.6798
F1-Score: 0.0081
Precision: 0.0040
Recall: 0.8667

Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.68      0.81      9985
           1       0.00      0.87      0.01        15

    accuracy                           0.68     10000
   macro avg       0.50      0.77      0.41     10000
weighted avg       1.00      0.68      0.81     10000



## Support Vector Machine (SVM)

In [22]:
from sklearn.svm import SVC

# Load Support Vector Classifier
# Note: probability=True is needed if you want to calculate AUC-ROC later
svm_clf = SVC()
# Fit it to the data
svm_clf.fit(X_train, y_train)

# Test on the test set
preds_svm = svm_clf.predict(X_test)

# Evaluate SVM
evaluate_model(preds_svm, y_test, model_name="SVM Classifier")


SVM Classifier Metrics:
Accuracy: 0.6787
F1-Score: 0.0000
Precision: 0.0000
Recall: 0.0000

Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.68      0.81     10000
           1       0.00      0.00      0.00         0

    accuracy                           0.68     10000
   macro avg       0.50      0.34      0.40     10000
weighted avg       1.00      0.68      0.81     10000



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: 